In [2]:
import pandas as pd
from prophet import Prophet
from statsmodels.tsa.arima.model import ARIMA

# Load dataset exported from Power BI
Forecast_Model ='https://raw.githubusercontent.com/Umangi123/Final-Year-Project_w1954804/refs/heads/main/data/Forecast%20Model.csv'
df = pd.read_csv(Forecast_Model)

df.head()

,Year,Total_Consumption_MT,Net_Harvested_ha,Annual_Yield_kg_per_ha
0,2004,1954122.214,3903,3250.576480
1,2005,2027717.357,5672,2989.245416
2,2006,2052133.601,3233,2974.017940
3,2007,2111551.837,3948,3472.644377
4,2008,2161339.933,7067,3378.236876


In [3]:
# Sort data by year
df = df.sort_values("Year")

df.tail()

,Year,Total_Consumption_MT,Net_Harvested_ha,Annual_Yield_kg_per_ha
15,2019,4317392.843,3104,3423.002577
16,2020,4698270.690,6012,3521.956088
17,2021,4747328.796,5235,2495.702006
18,2022,4274477.098,6028,2593.563371
19,2023,4235047.310,4983,3012.442304


In [4]:
# Prepare dataset for Prophet (needs ds and y format)
cons_df = df[["Year", "Total_Consumption_MT"]].copy()
cons_df.columns = ["ds", "y"]
cons_df["ds"] = pd.to_datetime(cons_df["ds"], format="%Y")

cons_df.head()

,ds,y
0,2004-01-01,1954122.214
1,2005-01-01,2027717.357
2,2006-01-01,2052133.601
3,2007-01-01,2111551.837
4,2008-01-01,2161339.933


In [5]:
# Train ARIMA model for yield
arima_yield_model = ARIMA(df["Annual_Yield_kg_per_ha"], order=(1,1,1)).fit()

# Train Prophet model for consumption
prophet_cons_model = Prophet()
prophet_cons_model.fit(cons_df)

# Generate forecasts
forecast_steps = 6

# Yield forecast (ARIMA)
yield_forecast_values = arima_yield_model.forecast(steps=forecast_steps).values

# Consumption forecast (Prophet)
future_cons = prophet_cons_model.make_future_dataframe(periods=forecast_steps, freq="YE")
cons_forecast = prophet_cons_model.predict(future_cons)
cons_forecast_values = cons_forecast["yhat"].tail(forecast_steps).values

/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/statespace/sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'
INFO:prophet:Disabling weekly seasonality. Run prophet with weekly_seasonality=True to override this.
INFO:prophet:Disabling daily seasonality. Run prophet with daily_seasonality=True to override this.
INFO:prophet:n_changepoints greater than number of observations. Using 15.


In [6]:
# Create future years list
last_year = df["Year"].max()
future_years = list(range(last_year + 1, last_year + forecast_steps + 1))

# Combine results into final table
forecast_df_1 = pd.DataFrame({
    "Year": future_years,
    "Forecast_Yield_kg_per_ha": yield_forecast_values,
    "Forecast_Consumption_MT": cons_forecast_values
})

print("\nForecast Output Table:\n")
print(forecast_df_1)

# Save & download the Excel
forecast_df_1.to_excel("forecast_output.xlsx", index=False)

from google.colab import files
files.download("forecast_output.xlsx")


Forecast Output Table:

   Year  Forecast_Yield_kg_per_ha  Forecast_Consumption_MT
0  2024               3117.385465             5.050928e+06
1  2025               3159.870866             5.178069e+06
2  2026               3177.070741             5.345927e+06
3  2027               3184.033973             5.526674e+06
4  2028               3186.852982             5.720261e+06
5  2029               3187.994236             5.847402e+06


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>